# Борьба с переобучением: подбор гиперпараметров Random Forest

**Задача:** классификация эмоций по речи (аудио) на датасете Dusha (4 класса: `angry`, `sad`, `neutral`, `positive`).

**Проблема:** Random Forest с гиперпараметрами по умолчанию (`max_depth=None`, `min_samples_split=2`, `min_samples_leaf=1`) строит деревья до полного разделения обучающей выборки и **запоминает данные**: `train accuracy ≈ 1.0` при `test accuracy ≈ 0.47` (см. отчёт `random_forest_dusha_resd_train_training_report.txt`).

**Цель ноутбука:**
1. Диагностировать переобучение (разрыв train/validation).
2. Пошагово подобрать гиперпараметры (число деревьев, `max_depth`, `min_samples_leaf`, `ccp_alpha`).
3. Убедиться, что разрыв сократился, а метрика на отложенном тесте выросла.
4. Зафиксировать результат: обновлённый конфиг + обученная модель + запись в `experiments.csv`.

Тестовая выборка **не участвует** ни в подборе, ни в валидации — она используется только для финальной проверки.


> **План**
> 1. Настройка окружения и загрузка данных
> 2. Стратифицированная подвыборка (для скорости подбора)
> 3. Диагностика переобучения: baseline (дефолтные параметры)
> 4. Шаг 1. Число деревьев `n_estimators` (кривая OOB)
> 5. Шаг 2. Глубина деревьев `max_depth`
> 6. Шаг 3. `min_samples_leaf` — главный регуляризатор
> 7. Шаг 4. Cost-complexity pruning `ccp_alpha`
> 8. Шаг 5. `RandomizedSearchCV` по суженной сетке
> 9. Итоговое обучение на полных данных + сравнение с baseline на тесте
> 10. Кривые обучения (learning curves)
> 11. Важность признаков
> 12. Выводы и воспроизведение

In [ ]:
import sys
from pathlib import Path

def _find_repo_root():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / 'ruintona' / 'my_experiments').is_dir():
            return p
    return None

_REPO_ROOT = _find_repo_root()
if _REPO_ROOT is not None and str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler

from ruintona.my_experiments.utils.config_utils import PROJECT_ROOT, DATASET_PATH
from ruintona.my_experiments.utils.lmdb_utils import load_audio_features_from_lmdb

AGGREGATED_DIR = DATASET_PATH / 'processed_dataset_090' / 'aggregated_dataset'
TRAIN_LMDB = AGGREGATED_DIR / 'dusha_resd_train.lmdb'
TEST_LMDB = AGGREGATED_DIR / 'dusha_resd_test.lmdb'

SEARCH_SUBSAMPLE = 20000   # подвыборка для подбора (полный train ~69k)
RANDOM_STATE = 42
CHECKPOINTS_AUDIO = PROJECT_ROOT / 'my_experiments' / 'checkpoints' / 'audio'

print('Train LMDB:', TRAIN_LMDB, '| exists:', TRAIN_LMDB.exists())
print('Test LMDB: ', TEST_LMDB, '| exists:', TEST_LMDB.exists())
print('Checkpoints:', CHECKPOINTS_AUDIO)


In [ ]:
def quick_report(model, X_tr, y_tr, X_val, y_val):
    """Быстрая сводка для диагностики переобучения."""
    tr_pred = model.predict(X_tr)
    val_pred = model.predict(X_val)
    oob = getattr(model, 'oob_score_', None)
    return {
        'train_acc': accuracy_score(y_tr, tr_pred),
        'oob_acc': float(oob) if oob is not None else np.nan,
        'val_acc': accuracy_score(y_val, val_pred),
        'val_f1_macro': f1_score(y_val, val_pred, average='macro', zero_division=0),
        'gap_train_val': accuracy_score(y_tr, tr_pred) - accuracy_score(y_val, val_pred),
    }


def fit_rf(**params):
    """Обёртка: RF с фиксированным random_state и OOB-оценкой."""
    return RandomForestClassifier(
        oob_score=True, random_state=RANDOM_STATE, n_jobs=-1, **params,
    )


## 1. Загрузка данных

Загружаем **полные** train/test. Для скорости подбора гиперпараметров делаем стратифицированную подвыборку `SEARCH_SUBSAMPLE` из train и режем её на `train/validation`. Полный train откладываем для финального обучения.

In [ ]:
print('Загрузка полного train...')
X_train_full, y_train_full = load_audio_features_from_lmdb(TRAIN_LMDB)
print('Загрузка test...')
X_test, y_test = load_audio_features_from_lmdb(TEST_LMDB)
print(f'Полный train: {X_train_full.shape}, test: {X_test.shape}')

# стратифицированная подвыборка для поиска
X_search, _, y_search, _ = train_test_split(
    X_train_full, y_train_full, train_size=SEARCH_SUBSAMPLE,
    stratify=y_train_full, random_state=RANDOM_STATE,
)

# holdout внутри поисковой выборки — для диагностики (validation)
X_search_tr, X_val, y_search_tr, y_val = train_test_split(
    X_search, y_search, test_size=0.2, stratify=y_search, random_state=RANDOM_STATE,
)

print(f'Подвыборка для поиска: {X_search.shape} '
      f'-> train {X_search_tr.shape}, val {X_val.shape}')
print('Распределение классов val:', {k: int(v) for k, v in zip(*np.unique(y_val, return_counts=True))})


## 2. Диагностика переобучения (baseline)

Обучаем Random Forest с **дефолтными** параметрами sklearn. Ожидание: `train_acc ≈ 1.0`, `val_acc` заметно ниже → модель запомнила обучающую выборку. Разрыв `train - val` — главный индикатор переобучения.

In [ ]:
base_params = dict(
    n_estimators=200, max_depth=None, min_samples_split=2,
    min_samples_leaf=1, max_features='sqrt',
)
model_default = fit_rf(**base_params)
model_default.fit(X_search_tr, y_search_tr)

res_default = quick_report(model_default, X_search_tr, y_search_tr, X_val, y_val)
print(f"""
BASELINE (дефолтные параметры sklearn):
  Train accuracy: {res_default['train_acc']:.4f}
  OOB accuracy:   {res_default['oob_acc']:.4f}
  Val accuracy:   {res_default['val_acc']:.4f}
  Val F1-macro:   {res_default['val_f1_macro']:.4f}
  GAP train-val:  {res_default['gap_train_val']:.4f}
""")
print('Глубины деревьев: min={}, max={}, mean={:.1f}'.format(
    min(t.get_depth() for t in model_default.estimators_),
    max(t.get_depth() for t in model_default.estimators_),
    np.mean([t.get_depth() for t in model_default.estimators_]),
))


> **Вывод по диагностике.** Если разрыв `train - val > 0.3` при `train_acc ≈ 1.0` — это классический признак переобучения: модель выучила шум/уникальные сэмплы вместо обобщающих закономерностей. Кривые ниже показывают, **как каждый гиперпараметр влияет на этот разрыв**.

## 3. Шаг 1. Число деревьев `n_estimators`

Больше деревьев = меньше дисперсия, но есть точка насыщения. Ищем её по OOB-оценке (бесплатная внутренняя валидация RF) — добавлять деревья после насыщения бессмысленно и дорого.

In [ ]:
n_estimators_range = [50, 100, 200, 300, 500]
train_scores, oob_scores = [], []
for n in n_estimators_range:
    m = fit_rf(n_estimators=n, max_depth=20, min_samples_leaf=4)
    m.fit(X_search_tr, y_search_tr)
    train_scores.append(accuracy_score(y_search_tr, m.predict(X_search_tr)))
    oob_scores.append(m.oob_score_)
    print(f'n_estimators={n:4d}: train_acc={train_scores[-1]:.4f}, oob_acc={oob_scores[-1]:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(n_estimators_range, train_scores, 'o-', label='train acc')
plt.plot(n_estimators_range, oob_scores, 's-', label='OOB acc')
plt.xlabel('n_estimators'); plt.ylabel('accuracy')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Влияние числа деревьев'); plt.tight_layout(); plt.show()


## 4. Шаг 2. Глубина деревьев `max_depth`

`max_depth=None` — деревья растут до конца и идеально запоминают train. Ограничение глубины — первый способ регуляризации. Смотрим, как меняется train/OOB/val при росте `max_depth`.

## 4. Шаг 2. Глубина деревьев `max_depth`

`max_depth=None` — деревья растут до конца и идеально запоминают train. Ограничение глубины — первый способ регуляризации. Смотрим, как меняется train/OOB/val при росте `max_depth`.

In [ ]:
max_depth_range = [None, 5, 10, 15, 20, 30]
rows = []
for d in max_depth_range:
    m = fit_rf(n_estimators=200, max_depth=d, min_samples_leaf=1)
    m.fit(X_search_tr, y_search_tr)
    r = quick_report(m, X_search_tr, y_search_tr, X_val, y_val)
    rows.append({'max_depth': 'None' if d is None else d, **r})
    print(f"max_depth={d}: train={r['train_acc']:.4f}, oob={r['oob_acc']:.4f}, val={r['val_acc']:.4f}, gap={r['gap_train_val']:.4f}")

df_depth = pd.DataFrame(rows)
plt.figure(figsize=(8, 4))
plt.plot(range(len(max_depth_range)), df_depth['train_acc'], 'o-', label='train acc')
plt.plot(range(len(max_depth_range)), df_depth['oob_acc'], 's-', label='OOB acc')
plt.plot(range(len(max_depth_range)), df_depth['val_acc'], '^-', label='val acc')
plt.xticks(range(len(max_depth_range)), [str(x) for x in max_depth_range])
plt.xlabel('max_depth'); plt.ylabel('accuracy')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Влияние глубины деревьев на переобучение'); plt.tight_layout(); plt.show()


## 5. Шаг 3. `min_samples_leaf` — главный регуляризатор

`min_samples_leaf` запрещает дереву создавать листья с меньшим числом сэмплов — это не даёт изолировать единичные (шумовые) примеры. Часто даёт больший выигрыш, чем ограничение глубины.

In [ ]:
min_samples_leaf_range = [1, 2, 4, 8, 16, 32]
rows = []
for msl in min_samples_leaf_range:
    m = fit_rf(n_estimators=200, max_depth=None, min_samples_leaf=msl)
    m.fit(X_search_tr, y_search_tr)
    r = quick_report(m, X_search_tr, y_search_tr, X_val, y_val)
    rows.append({'min_samples_leaf': msl, **r})
    print(f"min_samples_leaf={msl:3d}: train={r['train_acc']:.4f}, oob={r['oob_acc']:.4f}, val={r['val_acc']:.4f}, gap={r['gap_train_val']:.4f}")

df_leaf = pd.DataFrame(rows)
plt.figure(figsize=(8, 4))
plt.plot(min_samples_leaf_range, df_leaf['train_acc'], 'o-', label='train acc')
plt.plot(min_samples_leaf_range, df_leaf['oob_acc'], 's-', label='OOB acc')
plt.plot(min_samples_leaf_range, df_leaf['val_acc'], '^-', label='val acc')
plt.xlabel('min_samples_leaf'); plt.ylabel('accuracy')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Влияние min_samples_leaf на переобучение'); plt.tight_layout(); plt.show()


## 6. Шаг 4. Cost-complexity pruning `ccp_alpha`

Альтернатива ограничению глубины — пост-прунинг. `cost_complexity_pruning_path` даёт зависимость «качество → штраф за сложность `alpha`». Чем больше `alpha`, тем сильнее обрезаются деревья (регуляризация сильнее).

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Путь прунинга строим на одиночном дереве:
# у RandomForest в sklearn 1.8 нет cost_complexity_pruning_path,
# но параметр ccp_alpha передаётся в каждое дерево ансамбля.
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt.fit(X_search_tr, y_search_tr)
path = dt.cost_complexity_pruning_path(X_search_tr, y_search_tr)
alphas_full = path.ccp_alphas
print(f'Одиночное дерево: доступно alpha={len(alphas_full)}, '
      f'train_acc без прунинга={accuracy_score(y_search_tr, dt.predict(X_search_tr)):.4f}')

# Применяем pruning к RandomForest: sweep по нескольким значениям alpha
alpha_range = np.linspace(alphas_full[0], alphas_full[-1] * 0.5, 7)
rows = []
for a in alpha_range:
    m = fit_rf(n_estimators=200, max_depth=None, min_samples_leaf=1, ccp_alpha=float(a))
    m.fit(X_search_tr, y_search_tr)
    r = quick_report(m, X_search_tr, y_search_tr, X_val, y_val)
    rows.append({'ccp_alpha': float(a), **r})
    print(f"ccp_alpha={a:.6f}: train={r['train_acc']:.4f}, oob={r['oob_acc']:.4f}, val={r['val_acc']:.4f}, gap={r['gap_train_val']:.4f}")

df_alpha = pd.DataFrame(rows)
plt.figure(figsize=(8, 4))
plt.plot(df_alpha['ccp_alpha'], df_alpha['train_acc'], 'o-', label='train acc')
plt.plot(df_alpha['ccp_alpha'], df_alpha['oob_acc'], 's-', label='OOB acc')
plt.plot(df_alpha['ccp_alpha'], df_alpha['val_acc'], '^-', label='val acc')
plt.xlabel('ccp_alpha'); plt.ylabel('accuracy')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Влияние ccp_alpha (pruning)'); plt.tight_layout(); plt.show()


## 7. Шаг 5. `RandomizedSearchCV` по суженной сетке

Отдельные шаги дали понимание, но гиперпараметры взаимодействуют. Финальный шаг — `RandomizedSearchCV` по **суженной** области вокруг лучших значений, с `cv=3` и `scoring='f1_macro'`. Случайный поиск предпочтительнее полного перебора на таком объёме данных (быстрее, хорошо покрывает пространство).

In [ ]:
search_grid = {
    'n_estimators': [200, 300],
    'max_depth': [None, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [2, 4, 8],
    'max_features': ['sqrt', 'log2'],
    'ccp_alpha': [0.0, 0.0005, 0.001],
}

search = RandomizedSearchCV(
    fit_rf(),
    search_grid,
    n_iter=12, cv=3, scoring='f1_macro',
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1,
)
search.fit(X_search_tr, y_search_tr)

print(f"Лучшие параметры: {search.best_params_}")
print(f"Лучший CV f1-macro: {search.best_score_:.4f}")
print()
cv_df = pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
cv_df[['rank_test_score', 'mean_test_score', 'std_test_score', 'params']].head(10)


## 8. Итоговое обучение на полных данных + сравнение с baseline

- Лучшие параметры из поиска применяем к **полному train** (69k) и оцениваем на **тесте**.
- Для «до» используем ранее сохранённую модель `random_forest_dusha_resd_train_model.pkl` (дефолтные параметры, обучена на полном train).
- `evaluate_sklearn_classifier` выводит метрики на train **и** test — разрыв виден сразу.

In [ ]:
best_params = dict(search.best_params_)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)

model_tuned = fit_rf(**best_params)
model_tuned.fit(X_train_scaled, y_train_full)
print('Модель tuned обучена на полном train')
print(f'Глубины деревьев: min={min(t.get_depth() for t in model_tuned.estimators_)}, '
      f'max={max(t.get_depth() for t in model_tuned.estimators_)}, '
      f'mean={np.mean([t.get_depth() for t in model_tuned.estimators_]):.1f}')


In [ ]:
from ruintona.my_experiments.utils.model_io import load_sklearn_model
from ruintona.my_experiments.utils.sklearn_utils import evaluate_sklearn_classifier

# --- 'ДО': сохранённый baseline (дефолтные параметры) ---
try:
    model_old, scaler_old = load_sklearn_model(
        'dusha_resd_train', models_dir=CHECKPOINTS_AUDIO, model_name='random_forest',
    )
    print('\n########## BASELINE (дефолтные параметры) ##########')
    metrics_old = evaluate_sklearn_classifier(
        model_old, X_train_full, y_train_full, X_test, y_test,
        transform_fn=scaler_old.transform,
    )
except FileNotFoundError as e:
    print('Baseline-модель не найдена, пропускаем сравнение:', e)
    metrics_old = None

# --- 'ПОСЛЕ': tuned ---
print('\n########## TUNED (подобранные гиперпараметры) ##########')
metrics_tuned = evaluate_sklearn_classifier(
    model_tuned, X_train_full, y_train_full, X_test, y_test,
    transform_fn=scaler.transform,
)


In [ ]:
def summary_row(name, m):
    return {
        'model': name,
        'train_acc': round(m['train_accuracy'], 4),
        'test_acc': round(m['test_accuracy'], 4),
        'test_f1_macro': round(m['test_f1_macro'], 4),
        'gap_train_test': round(m['overfit_gap_accuracy'], 4),
    }

rows = []
if metrics_old is not None:
    rows.append(summary_row('RF baseline (default params)', metrics_old))
rows.append(summary_row('RF tuned (RandomizedSearchCV)', metrics_tuned))

summary = pd.DataFrame(rows)
summary


## 9. Кривые обучения (learning curves)

Кривая обучения показывает, как растёт качество с объёмом данных. Если train-кривая заметно выше валидационной и обе ещё не вышли на плато — модель переобучается, но добавлять данные по-прежнему полезно.

In [ ]:
train_sizes, train_scores_cv, val_scores_cv = learning_curve(
    fit_rf(**best_params),
    X_search, y_search,
    cv=3, scoring='f1_macro',
    train_sizes=np.linspace(0.1, 1.0, 5),
    n_jobs=-1, random_state=RANDOM_STATE,
)

train_mean = train_scores_cv.mean(axis=1)
val_mean = val_scores_cv.mean(axis=1)
train_std = train_scores_cv.std(axis=1)
val_std = val_scores_cv.std(axis=1)

plt.figure(figsize=(8, 4.5))
plt.plot(train_sizes, train_mean, 'o-', label='train f1-macro')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
plt.plot(train_sizes, val_mean, 's-', label='CV f1-macro')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15)
plt.xlabel('Размер обучающей выборки'); plt.ylabel('f1-macro')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Learning curves (tuned RF)'); plt.tight_layout(); plt.show()

for size, t, v in zip(train_sizes, train_mean, val_mean):
    print(f'size={int(size):5d}: train_f1={t:.4f}, cv_f1={v:.4f}')


## 10. Важность признаков

Признаки: первые 64 — `mel mean` (среднее по кадрам каждого мел-бина), вторые 64 — `mel std`. Смотрим, какие признаки реально используются моделью (после регуляризации важность становится устойчивее).

In [ ]:
importances = model_tuned.feature_importances_
top_idx = np.argsort(importances)[-20:]
labels = [f'mel_mean[{i}]' if i < 64 else f'mel_std[{i-64}]' for i in top_idx]

print('Доля важности: mel_mean=%.3f, mel_std=%.3f' % (importances[:64].sum(), importances[64:].sum()))

plt.figure(figsize=(8, 6))
plt.barh(range(len(top_idx)), importances[top_idx], color='#2a9d8f')
plt.yticks(range(len(top_idx)), labels)
plt.xlabel('feature importance')
plt.title('Топ-20 признаков (tuned RF)')
plt.tight_layout(); plt.show()


## 11. Сохранение результата

Сохраняем tuned-модель отдельно (не затираем baseline), чтобы конфиг `configs/audio/random_forest_tuned.json` можно было воспроизвести из CLI, а строка эксперимента попала в `experiments.csv`.

In [ ]:
from ruintona.my_experiments.utils.model_io import save_sklearn_model
from ruintona.my_experiments.utils.config_utils import models_dir_for

models_dir = models_dir_for(Path('ruintona/my_experiments/audio_models/baseline/random_forest.py'))
save_sklearn_model(
    model_tuned, scaler, 'dusha_resd_train',
    models_dir=models_dir, model_name='random_forest_tuned',
    training_params={
        **best_params, 'criterion': model_tuned.criterion,
        'oob_score': True, 'random_state': RANDOM_STATE,
        'search': 'RandomizedSearchCV (cv=3, f1_macro)',
        'search_n_iter': search.n_iter,
        'search_best_cv_f1_macro': float(search.best_score_),
    },
    test_metrics=metrics_tuned,
)
print('\nTuned-конфиг: ruintona/configs/audio/random_forest_tuned.json')
print('Воспроизведение из CLI:')
print('  poetry run python ruintona/my_experiments/audio_models/baseline/random_forest.py \\')
print('      --mode train --config audio/random_forest_tuned.json')
print('  # или полный подбор:')
print('  poetry run python ruintona/my_experiments/audio_models/baseline/random_forest.py \\')
print('      --mode tune --max-samples 20000 --search-iterations 12 --cv-folds 3')


## 12. Выводы

1. **Проблема подтверждена:** baseline RF с дефолтными параметрами показывает `train acc ≈ 1.0` при тестовом `≈ 0.47` — модель запоминает данные, разрыв `train - test` большой.
2. **Диагностика:** OOB-оценка и разрыв `train/val` позволяют следить за переобучением на каждой итерации без использования теста.
3. **Регуляризация:** ограничение `max_depth`, увеличение `min_samples_leaf` и `ccp_alpha` сокращают разрыв, почти не теряя в качестве на валидации.
4. **Автоматизация:** `RandomizedSearchCV` (scoring `f1_macro`, `cv=3`) на суженной сетке собрал лучшие значения гиперпараметров; финальная модель обучена на полном train.
5. **Честная оценка:** test использовался один раз — только для финального сравнения. Разрыв `train/test` теперь явно выводится в отчёте каждой sklearn-модели (`ДИАГНОСТИКА ПЕРЕОБУЧЕНИЯ`).